In [ ]:
from pathlib import Path
import shutil
import pandas as pd

import teehr
from teehr import RemoteReadWriteEvaluation

from utils.setup_utils import create_minio_spark_session
import utils.cbrfc_wsv_seasonal_utils as cbrfc_wsv_seasonal_utils

### Setup eval

In [ ]:
# create spark session
spark = create_minio_spark_session()

In [ ]:
# define evaluation
ev = RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

In [ ]:
# configure download
ev.download.configure(
    api_key='your_key_here'
)

### Define new crosswalk and POR

In [ ]:
# setup a crosswalk between CBRFC obs and CBRFC forecasts for seasonal wsv data (same locs as cbrfc_esp_wsv)
cbrfc_to_cbrfc_dict = {
    'cbrfc-dlac2': 'cbrfc-dlac2', # DLAC2 in CBRFC 
    'cbrfc-drgc2': 'cbrfc-drgc2', # DRGC2 in CBRFC
    'cbrfc-ydlc2': 'cbrfc-ydlc2', # YDLC2 in CBRFC
}

# define water year to obtain seasonal forecasts
water_year = 2026

### Update domain tables ahead of timeseries ingest

In [ ]:
# add new primary_locations to the locations table using existing usgs entries for now
existing_locs_gdf = ev.locations.to_geopandas()
new_locs = [
    {
        'id':'cbrfc-dlac2',
        'name':existing_locs_gdf.loc[existing_locs_gdf['id']=='usgs-09149500', 'name'].values[0], 
        'geometry':existing_locs_gdf.loc[existing_locs_gdf['id']=='usgs-09149500', 'geometry'].values[0],
    },
    {
        'id':'cbrfc-drgc2',
        'name':existing_locs_gdf.loc[existing_locs_gdf['id']=='usgs-09361500', 'name'].values[0], 
        'geometry':existing_locs_gdf.loc[existing_locs_gdf['id']=='usgs-09361500', 'geometry'].values[0],
    },
    {
        'id':'cbrfc-ydlc2',
        'name':existing_locs_gdf.loc[existing_locs_gdf['id']=='usgs-09260050', 'name'].values[0], 
        'geometry':existing_locs_gdf.loc[existing_locs_gdf['id']=='usgs-09260050', 'geometry'].values[0],
    },
]
import geopandas as gpd
new_locs_gdf = gpd.GeoDataFrame(new_locs, crs=existing_locs_gdf.crs)
ev.locations.load_dataframe(new_locs_gdf)


# add configurations for cbrfc seasonal wsv data to the configurations table
from teehr import Configuration
configuration = Configuration(
    name="cbrfc_seasonal_wsv_forecast",
    timeseries_type="secondary",
    description="CBRFC seasonal water-supply volume forecasts",
)
ev.configurations.add(configuration)
configuration = Configuration(
    name="cbrfc_seasonal_wsv_observed",
    timeseries_type="primary",
    description="CBRFC seasonal water-supply volume observations",
)
ev.configurations.add(configuration)


# add unit for volume
from teehr import Unit
unit = Unit(
    name='m^3',
    long_name='Cubic Meter'
)
ev.units.add(unit)


# add variables for seasonal wsv data to the variables table
variable_mapping = {
	'wsv_esp_max':'ESP cumulative water-supply volume forecast (MAX)',
	'wsv_esp_p10':'ESP cumulative water-supply volume forecast (P10)',
	'wsv_esp_p30':'ESP cumulative water-supply volume forecast (P30)',
	'wsv_esp_p50':'ESP cumulative water-supply volume forecast (P50)',
	'wsv_esp_p70':'ESP cumulative water-supply volume forecast (P70)',
	'wsv_esp_p90':'ESP cumulative water-supply volume forecast (P90)',
	'wsv_esp_min':'ESP cumulative water-supply volume forecast (MIN)',
	'wsv_official_crx':'Official cumulative water-supply volume forecast (CRX)',
	'wsv_official_c30':'Official cumulative water-supply volume forecast (C30)',
	'wsv_official_cmp':'Official cumulative water-supply volume forecast (CMP)',
	'wsv_official_c70':'Official cumulative water-supply volume forecast (C70)',
	'wsv_official_crn':'Official cumulative water-supply volume forecast (CRN)',
	'wsv_obs':'Observed cumulative water-supply volume',
}
from teehr import Variable
for key in variable_mapping.keys():
    variable = Variable(
        name=key,
        long_name=variable_mapping[key],
    )
    ev.variables.add(variable)


# add crosswalk for cbrfc seasonal wsv data to the crosswalk table
crosswalk_dict = {
    'primary_location_id': cbrfc_to_cbrfc_dict.keys(),
    'secondary_location_id': cbrfc_to_cbrfc_dict.values()
}
temp = pd.DataFrame(crosswalk_dict)
ev.location_crosswalks.load_dataframe(temp)

### Query the cbrfc seasonal wsv data

In [ ]:
primary_locs = list(cbrfc_to_cbrfc_dict.keys())
sim_df, obs_df = cbrfc_wsv_seasonal_utils.fetch_seasonal_wsup_forecasts(primary_locs, year=water_year)

### Write the timeseries data to the eval

In [ ]:
ev.primary_timeseries.load_dataframe(df=obs_df)

In [ ]:
ev.secondary_timeseries.load_dataframe(df=sim_df)

### Peek at the data (optional QC -- feel free to strip this out)

In [ ]:
import matplotlib.pyplot as plt

def plot_seasonal_wsup_qc(location_id, obs_df, sim_df, plot_start):
    kaf_to_cubic_meters = 1000 * 1233.48184

    plot_sim_df = sim_df[sim_df["location_id"].eq(location_id)].copy()
    plot_obs_df = obs_df[obs_df["location_id"].eq(location_id)].copy()
    plot_sim_df["value"] = plot_sim_df["value"] / kaf_to_cubic_meters
    plot_obs_df["value"] = plot_obs_df["value"] / kaf_to_cubic_meters

    esp_p90 = plot_sim_df[plot_sim_df["variable_name"].eq("wsv_esp_p90")].sort_values("value_time")
    esp_p50 = plot_sim_df[plot_sim_df["variable_name"].eq("wsv_esp_p50")].sort_values("value_time")
    esp_p10 = plot_sim_df[plot_sim_df["variable_name"].eq("wsv_esp_p10")].sort_values("value_time")
    official = plot_sim_df[plot_sim_df["variable_name"].str.startswith("wsv_official")].sort_values("value_time").copy()
    obs = plot_obs_df[plot_obs_df["variable_name"].eq("wsv_obs")].sort_values("value_time")

    official["plot_time"] = official["value_time"].dt.to_period("M").dt.to_timestamp() + pd.offsets.Day(14)

    esp_envelope = (
        esp_p10[["value_time", "value"]]
        .rename(columns={"value": "p10"})
        .merge(
            esp_p90[["value_time", "value"]].rename(columns={"value": "p90"}),
            on="value_time",
            how="inner",
        )
    )

    official_wide = official.pivot_table(
        index="plot_time",
        columns="variable_name",
        values="value",
        aggfunc="first",
    )
    official_value_columns = [column for column in official_wide.columns if column.startswith("wsv_official")]
    official_whiskers = official_wide[official_wide["wsv_official_cmp"].notna()].copy()
    official_whiskers = official_whiskers[official_whiskers[official_value_columns].count(axis=1).gt(1)]
    official_whiskers["low"] = official_whiskers[official_value_columns].min(axis=1)
    official_whiskers["high"] = official_whiskers[official_value_columns].max(axis=1)
    official_whiskers["lower_error"] = official_whiskers["wsv_official_cmp"].sub(official_whiskers["low"]).clip(lower=0)
    official_whiskers["upper_error"] = official_whiskers["high"].sub(official_whiskers["wsv_official_cmp"]).clip(lower=0)
    official_dashes = official[official["variable_name"].ne("wsv_official_cmp")]

    fig, ax = plt.subplots(figsize=(11, 6))
    ax.fill_between(
        esp_envelope["value_time"],
        esp_envelope["p10"],
        esp_envelope["p90"],
        color="tab:blue",
        alpha=0.15,
        label="wsv_esp_p10 to wsv_esp_p90",
    )
    ax.plot(esp_p50["value_time"], esp_p50["value"], color="tab:blue", linewidth=2, label="wsv_esp_p50")
    ax.errorbar(
        official_whiskers.index,
        official_whiskers["wsv_official_cmp"],
        yerr=[official_whiskers["lower_error"], official_whiskers["upper_error"]],
        fmt="o",
        color="deeppink",
        ecolor="hotpink",
        elinewidth=1.5,
        capsize=4,
        markersize=5,
        alpha=0.85,
        label="wsv_official_cmp",
    )
    ax.scatter(
        official_dashes["plot_time"],
        official_dashes["value"],
        marker="_",
        color="deeppink",
        s=180,
        linewidths=1.5,
        alpha=0.85,
        label="wsv_official components",
    )
    ax.plot(obs["value_time"], obs["value"], color="tab:red", linewidth=2, label="wsv_obs")

    ax.set_xlim(left=plot_start)
    ax.set_title(f"CBRFC seasonal WSV forecast QC: {location_id}")
    ax.set_xlabel("Value time")
    ax.set_ylabel("Water supply volume (kaf)")
    ax.legend()
    ax.grid(True, alpha=0.25)
    fig.autofmt_xdate()
    plt.show()

In [ ]:
plot_start = pd.Timestamp("2025-12-16")
plot_seasonal_wsup_qc(location_id=f"{primary_locs[0]}", obs_df=obs_df, sim_df=sim_df, plot_start=plot_start)
plot_seasonal_wsup_qc(location_id=f"{primary_locs[1]}", obs_df=obs_df, sim_df=sim_df, plot_start=plot_start)
plot_seasonal_wsup_qc(location_id=f"{primary_locs[2]}", obs_df=obs_df, sim_df=sim_df, plot_start=plot_start)

### Kill spark

In [ ]:
spark.stop()